In [19]:
import numpy as np
import marimo as mo
import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl
from skimpy import skim
import sweetviz as sv
from data_profiling import ProfileReport
import pygwalker as pyg
import polars.selectors as cs

from pathlib import Path

In [4]:
base_path = Path("/Users/platon/Documents/kaggle-house")

In [5]:
train_dataset = pl.read_csv(base_path / 'data/train.csv', null_values="NA")
test_dataset= pl.read_csv(base_path / 'data/test.csv', null_values='NA')

In [ ]:
train_dataset.to_pandas().hist(figsize=(16, 12), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
skimpy_summary = skim(train_dataset)

skimpy — малочитабельно для столь обширного кол-ва фичей. но можно быстро пробежаться и посмотреьт еще и на категориальные

In [6]:
train_pd = train_dataset.to_pandas()
test_pd = test_dataset.to_pandas()

In [ ]:
report_dp = ProfileReport(train_pd, title="House Prices EDA")
report_dp.to_file("eda_profile.html")

In [ ]:
report_sv = sv.analyze(train_pd, target_feat="SalePrice")

In [ ]:
report_sv.show_html('eda_profile_sv.html')

In [ ]:
report_compare = sv.compare(
    [train_pd.drop(columns=["SalePrice", "Id"]), "Train"],
    [test_pd.drop(columns=["Id"]), "Test"],
)
report_compare.show_html("eda_train_test.html")

- фича Alley — там чет 94% NA
- фича Street — там чет 99% одного значения
- фича Utilities — там чет 99% одного значения
- фича LandSlop - 95% одного значения и еще два других значения по 4% и 1%
- фича Condition2 — 99% одного значения и еще 7 других значений по 1%
- фича RoofMatl — 98% одного значения и еще 7 других значений по 1%, но есть какая-то корреляция с SalePrice = 0.18
- фича BsmtFinSF2 — чет просто странная, там 88% нулей, а остальные значения почти все разные, диапазон от 0 до 1474
- фича BsmtHalfBath — там 94% нулей, а остальные странные тоже, думаю точно удалять стоит
- фича FireplaceQu — там 47% NA, нужно чем-то заполнять, но как? может просто 0, а потом сделать категориальную фичу есть камин или нет
- фича EnclosedPorch, 3SsnPorch, ScreenPorch — там >85% одно значение, а остальные раскиданы сильно, надо подумать, мб просто добавить фичу, есть веранда или нет, а потом сделать категориальную фичу, что за веранда
- фича PoolArea — там 99% нулей, просто в категориальную фичу, есть бассейн или нет
- фича PoolQC - там 99% NA нах
- фича Fence — там 81% NA, нужно чем-то заполнять, но как? может просто "IDK", а потом сделать категориальную фичу есть забор или нет
- фича MiscFeature - там 96% NA, как будто просто удалить
- фича MiscVal — там 96.4% нулей, как будто делит
- фича YrSold — чуть бесполезная фича вроде, там везде одна цена по сути 181к +- 4к, можно делит?

гпт сказал:

| Признак | Моя оценка | Что делать |
|---|---|---|
| `Alley` | 94% NaN означают отсутствие доступа к переулку, а не неизвестное значение | Оставить, заполнить `NoAlley` |
| `Street` | Всего 6 домов с гравийной дорогой | Кандидат на удаление |
| `Utilities` | Только 1 дом отличается от остальных | Проверить удаление через CV |
| `LandSlope` | 95% `Gtl`, но остальные 78 домов имеют иной уклон. Связь с `LotArea` — 0.55 | Оставить |
| `Condition2` | Только 15 домов имеют значение, отличное от `Norm` | Кандидат на удаление |
| `RoofMatl` | 98% `CompShg`, связь с ценой 0.18 | Оставить для проверки, возможно объединить редкие категории |
| `BsmtFinSF2` | 89% нулей, остальные значения — площадь второй отделанной части подвала | Оставить числовым |
| `BsmtHalfBath` | 94% нулей, количество дополнительных полуванных в подвале | Оставить, проверить объединение с другими ванными |
| `FireplaceQu` | 47% NaN соответствуют домам без камина | Оставить, заполнить `NoFireplace` |
| `EnclosedPorch`, `3SsnPorch`, `ScreenPorch` | Разные типы веранд, с большой долей нулей | Оставить, добавить агрегированные признаки |
| `PoolArea` | Бассейн есть только у 7 домов | Проверить бинарный `HasPool` вместо площади |
| `PoolQC` | Пропуски означают отсутствие бассейна | Заполнить `NoPool`, проверить полезность |
| `Fence` | 81% NaN означают отсутствие забора | Оставить, заполнить `NoFence` |
| `MiscFeature` | 96% NaN означают отсутствие дополнительных объектов | Кандидат на удаление |
| `MiscVal` | 96.4% нулей, слабая линейная связь с ценой −0.02 | Кандидат на удаление, проверить вместе с `MiscFeature` |
| `YrSold` | Средние цены по годам близки, но это не исключает временного эффекта | Пока оставить |


In [23]:
categorical = train_dataset.select(cs.string() | cs.categorical()).columns
categorical

['MSZoning',
 'Street',
 'Alley',
 'LotShape',
 'LandContour',
 'Utilities',
 'LotConfig',
 'LandSlope',
 'Neighborhood',
 'Condition1',
 'Condition2',
 'BldgType',
 'HouseStyle',
 'RoofStyle',
 'RoofMatl',
 'Exterior1st',
 'Exterior2nd',
 'MasVnrType',
 'ExterQual',
 'ExterCond',
 'Foundation',
 'BsmtQual',
 'BsmtCond',
 'BsmtExposure',
 'BsmtFinType1',
 'BsmtFinType2',
 'Heating',
 'HeatingQC',
 'CentralAir',
 'Electrical',
 'KitchenQual',
 'Functional',
 'FireplaceQu',
 'GarageType',
 'GarageFinish',
 'GarageQual',
 'GarageCond',
 'PavedDrive',
 'PoolQC',
 'Fence',
 'MiscFeature',
 'SaleType',
 'SaleCondition']

In [24]:
numeric = train_dataset.select(cs.numeric()).columns
numeric

['Id',
 'MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold',
 'SalePrice']